In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard, CSVLogger

PROJECT_DIR = '/Users/erikdalgard/Library/CloudStorage/GoogleDrive-dalgard.erik@gmail.com/My Drive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks'
sys.path.append(PROJECT_DIR)
os.chdir(PROJECT_DIR)

In [ ]:
from models_2D import get_archi_1_2D, get_archi_2_2D, get_archi_3_2D
from models_3D import get_archi_1_3D, get_archi_2_3D, get_archi_3_3D
from models_2D import get_archi_1_2D, get_archi_2_2D, get_archi_3_2D

In [5]:
def train_network(model, X_train, y_train, X_val, y_val, project_dir, epochs=50, batch_size=64):
    """
    Compiles, logs and trains a given keras model. Saves the best model and training history to the project directory.

    Parameters:
        model (keras.Model): The uncompiled Keras model to be trained.
        X_train, y_train: Training features and labels
        X_val, y_val: Validation features and labels
        project_dir (str): Directory for saving assets
        epochs (int): Maximum number of training iterations
        batch_size (int): Number of samples per training batch
    """

    # Compiling the module with class imbalance awareness
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='sensitivity'),
            keras.metrics.AUC(name='auc')
        ]
    )
    
    # Create a subfolder for each model name
    model_folder = os.path.join(project_dir, model.name)
    os.makedirs(model_folder, exist_ok=True)
    
    checkpoint_path = os.path.join(model_folder, 'best_model.keras') # Updated extension to .keras
    log_path = os.path.join(model_folder, 'training_log.csv')

    # Setting up callbacks
    callbacks = [
        ModelCheckpoint(filepath=checkpoint_path, monitor='val_loss', save_best_only=True),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        CSVLogger(log_path),
    ]

    # Training the model
    print(f"\nTraining model: {model.name}")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,  # <-- Added missing comma here
        verbose=1
    )

    print(f"\nSuccessfully finished training {model.name}!")
    print(f"-> Best weights secured at: {checkpoint_path}")
    print(f"-> History saved to: {log_path}")

    return history